# Kiểm chứng chéo và sàng lọc ứng viên Hidden Meta
Đọc luật xuất từ notebook 05 và phân hoạch xuất từ 07. Mỗi tổ hợp được đếm trên dữ liệu đã làm sạch, bao gồm cả thắng lẫn thua. Đây là thống kê khám phá trên cùng dữ liệu, **chưa đánh giá holdout**; không khẳng định quan hệ nhân quả hay đội hình hoàn chỉnh.

Tiêu chí thử nghiệm: pick-rate ≤ 5%, Top 4 ≥ 60%, ít nhất 100 bản ghi. Cùng cộng đồng là thông tin đối chiếu, không bắt buộc để được coi là ứng viên. Ngưỡng không được tuyên bố là tối ưu. Không suy luận trang bị tối ưu từ luật chỉ chứa tướng.

In [14]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
from src.hidden_meta_verifier import run_verification
measurements, candidates, by_rank = run_verification(max_frequency=0.05, min_winrate=0.60, min_count=100)

{
  "records": 399754,
  "baseline_top4": 0.4999849907693231,
  "input_rules": 3654,
  "unique_sets": 961,
  "candidates": 374,
  "max_pick_rate": 0.05,
  "min_top4_rate": 0.6,
  "min_observations": 100,
  "graph_min_weight": 0.55,
  "source_sha256": "cb587a3b37bf21e86920f50f06f917d26aafc686683248661bee03e5979debff",
  "scope": "Descriptive full-data screening; no independent holdout; candidate subset from TV1 mining; not exhaustive discovery or causal evidence"
}


## Kết quả đếm trực tiếp
`observations`: số đội hình chứa đủ tướng; `pick_rate`: số đó chia tổng bản ghi; `top4_rate`: số Top 4 chia số quan sát. Lift chỉ thuộc luật đại diện trên mẫu thắng. Một tổ hợp được đếm một lần dù có nhiều hướng luật.

In [15]:
cols = ['champions','observations','top4_count','pick_rate','top4_rate','same_louvain','same_leiden']
display(candidates[cols].head(20).style.format({'pick_rate':'{:.2%}', 'top4_rate':'{:.2%}'}))
if candidates.empty: print('Không có ứng viên đạt đủ tiêu chí; không thêm số liệu minh họa.')

,champions,observations,top4_count,pick_rate,top4_rate,same_louvain,same_leiden
550,AurelionSol | Graves | MissFortune | Vi,8916,7667,2.23%,85.99%,True,True
545,AurelionSol | ChoGath | Graves | MissFortune,8979,7714,2.25%,85.91%,True,True
575,AurelionSol | Blitzcrank | Graves | MissFortune,9008,7721,2.25%,85.71%,True,True
584,AurelionSol | Graves | Malphite | MissFortune,9033,7734,2.26%,85.62%,False,False
540,AurelionSol | Ezreal | Graves | MissFortune,9019,7708,2.26%,85.46%,False,False
630,AurelionSol | Graves | Jinx | MissFortune,9147,7804,2.29%,85.32%,True,True
698,AurelionSol | Ezreal | MissFortune | Vi,10767,9165,2.69%,85.12%,False,False
799,AurelionSol | Graves | MissFortune,9216,7843,2.31%,85.10%,True,True
601,AurelionSol | ChoGath | Ezreal | MissFortune,10840,9220,2.71%,85.06%,False,False
587,AurelionSol | ChoGath | MissFortune | Vi,11181,9495,2.80%,84.92%,True,True


## Đối chiếu cộng đồng thật
Chỉ đánh dấu cùng cụm khi TẤT CẢ tướng của tổ hợp đều có trong phân hoạch và cùng một mã cộng đồng.

In [16]:
display(pd.DataFrame({'scope':['Toàn bộ tập ứng viên ARM','Các tập qua lọc tần suất và Top 4'], 'sets':[len(measurements),len(candidates)], 'same_louvain':[int(measurements.same_louvain.sum()),int(candidates.same_louvain.sum())], 'same_leiden':[int(measurements.same_leiden.sum()),int(candidates.same_leiden.sum())]}))

,scope,sets,same_louvain,same_leiden
0,Toàn bộ tập ứng viên ARM,961,592,592
1,Các tập qua lọc tần suất và Top 4,374,235,235


## Kiểm tra theo nhóm rank
Giữ nhãn tier của file nguồn. Tần suất theo rank có mẫu số là tổng bản ghi của nhóm rank đó. Các nhóm ít quan sát không đủ để khẳng định hiệu quả ổn định.

In [17]:
if not candidates.empty:
    selected=candidates.iloc[0].champions
    print('Tổ hợp đứng đầu:',selected)
    display(by_rank[by_rank.champions.eq(selected)].style.format({'pick_rate':'{:.2%}', 'top4_rate':'{:.2%}'}))

Tổ hợp đứng đầu: AurelionSol | Graves | MissFortune | Vi


,champions,tier,observations,top4_count,pick_rate,top4_rate
2750,AurelionSol | Graves | MissFortune | Vi,Challenger,2195,1847,2.74%,84.15%
2751,AurelionSol | Graves | MissFortune | Vi,Diamond,1659,1411,2.07%,85.05%
2752,AurelionSol | Graves | MissFortune | Vi,GrandMaster,2553,2204,3.19%,86.33%
2753,AurelionSol | Graves | MissFortune | Vi,Master,1944,1708,2.43%,87.86%
2754,AurelionSol | Graves | MissFortune | Vi,Platinum,565,497,0.71%,87.96%


In [18]:
fig, ax = plt.subplots(figsize=(9,5))
ax.scatter(measurements.pick_rate,measurements.top4_rate,s=12,alpha=.4,label='Các tổ hợp ARM')
ax.scatter(candidates.pick_rate,candidates.top4_rate,s=16,label='Ứng viên qua lọc')
ax.axvline(.05,color='gray',ls='--');ax.axhline(.60,color='red',ls='--')
ax.set(xlabel='Frequency across all player-game records',ylabel='Observed Top-4 rate',title='Exploratory candidate screening; no independent holdout')
ax.legend();fig.tight_layout();fig.savefig('../reports/hidden_meta_verified.png',dpi=140);plt.show()

## Hạn chế cần trình bày
- Chỉ xét tập ứng viên do ARM trên mẫu thắng đề xuất, không tìm hết mọi tổ hợp hiếm.
- Cùng dữ liệu được dùng để khám phá và đánh giá; tỷ lệ này chưa chứng minh khả năng tổng quát hóa.
- Dữ liệu có nhiều loại tướng/tộc hệ; cần xác minh set/patch trước khi gọi là meta của một mùa.
- Ảnh hưởng của rank, cấp độ, trang bị, sao và thời điểm chưa được kiểm soát.
- Khoảng Wilson chỉ mang tính tham khảo, chưa điều chỉnh phụ thuộc giữa người chơi cùng trận và nhiều phép thử.